In [4]:
import boto3
import json
import pandas as pd
import re
import time
import botocore
import os
from sklearn.metrics import classification_report, accuracy_score, mean_absolute_error, mean_squared_error, r2_score
from io import StringIO

In [5]:
# Definir a config para tentar impedir timeout
config = botocore.config.Config(
    retries={'max_attempts': 10},
    read_timeout=600,
    connect_timeout=300
)

In [6]:
# Iniciar clientes S3 e SageMaker
s3_client = boto3.client('s3', region_name='eu-west-1')
sagemaker_runtime = boto3.client('sagemaker-runtime', region_name="eu-west-1", config=config)

In [7]:
# Ler o dataset do S3
bucket_name = 'i32419'
obj = s3_client.get_object(Bucket=bucket_name, Key='datasets/Edmunds_Car_Ratings_final.csv')
csv_content = obj['Body'].read().decode('utf-8')
df = pd.read_csv(StringIO(csv_content))

In [8]:
# Endpoint fornecido
endpoint_name = 'meta-textgenerationneuron-llama-3-2-1b-2025-07-11-20-51-32-569'

In [41]:
# Inferência Zero Shot

# Função de inferência para sentimento
def classify_sentiment(review, fallback=False):
    """
    Envia uma requisição ao modelo para classificar o sentimento da review.
    Se 'fallback' for True, usa um prompt mais direto. Foi necessário devido às falhas do modelo (reviews não classificadas como Positive, Neutral e Negative que nós identificamos como Unknown)
    """
    if not fallback:
        prompt = f"""Classify the sentiment of the following review as one of the following three words only: Positive, Neutral or Negative. Answer with exactly one of those words.

    Review: {review}

    Sentiment:"""
    else:
        prompt = f"""Is the sentiment of this review Positive, Neutral or Negative? Respond only with one of those words.
    
Review: {review}
    
Sentiment:"""

    # Enviar requisição ao endpoint
    response = sagemaker_runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',
        Body=json.dumps({
            'inputs': prompt,
            'parameters': {
                'max_new_tokens': 5,
                'temperature': 0.1,
            }
        })
    )

    # Ler e decodificar a resposta
    result = response['Body'].read().decode('utf-8').strip()
    result_json = json.loads(result)
    generated = result_json['generated_text'].strip()

    # Tentar extrair o sentimento com regex
    match = re.search(r'\b(positive|neutral|negative)\b', generated, re.IGNORECASE)
    label = match.group(1).strip().capitalize() if match else "Unknown"

    return label, generated

# Função de inferência para rating
def predict_rating(review):
    prompt = f"""Predict a rating from 1.000 to 5.000, with 3 decimal places, for the following review. Answer with exactly four digits.
    
    Review: {review}
    
    Rating:"""

    # Enviar requisição ao endpoint
    response = sagemaker_runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',
        Body=json.dumps({
            'inputs': prompt,
            'parameters': {
                'max_new_tokens': 5,
                'temperature': 0.1,
            }
        })
    )

    # Ler e decodificar a resposta
    result = response['Body'].read().decode('utf-8').strip()
    result_json = json.loads(result)
    generated = result_json['generated_text'].strip()

    # Tentar extrair o rating com regex
    match = re.search(r'\b(\d+\.?\d{0,3})\b', generated)
    rating = float(match.group(1)) if match else None
    if rating is not None:
        rating = max(1.0, min(5.0, rating))

    return rating, generated


# Listas para armazenar os resultados
sentiments = []
ratings = []

In [42]:
# Loop principal
for idx, row in df.iterrows():
    review = row["Review"]

    # Classificar sentimento
    label, generated_sentiment = classify_sentiment(review)
    if label == "Unknown":
        label, generated_sentiment = classify_sentiment(review, fallback=True)

    # Prever rating
    rating, generated_rating = predict_rating(review)

    # Guardar os resultados
    sentiments.append(label)
    ratings.append(rating if rating is not None else "Unknown")

    # Print de progresso com raw output
    print(f"[{idx+1}/{len(df)}] Review: {review[:50]}... → Sentimento: {label}, Rating: {rating}")

    # Pequena pausa para evitar throttling
    time.sleep(0.1)

# Adicionar os resultados ao DataFrame
df["Predicted_Label"] = sentiments
df["Predicted_Rating"] = ratings

[1/11756] Review:  With the expected arrival of our 6th child, our T... → Sentimento: Positive, Rating: 3.5
[2/11756] Review:  Rear ac blow to slow that my kid do not want to b... → Sentimento: Positive, Rating: 3.0
[3/11756] Review:  This is not a small astro van type.You will need ... → Sentimento: Negative, Rating: 3.0
[4/11756] Review:  I am very satisfied with my  2014 Nissan NV SL. I... → Sentimento: Positive, Rating: 5.0


KeyboardInterrupt: 

In [10]:
# Avaliação

# Remover linhas com "Unknown" para avaliação do sentimento
df_filtered_sentiment = df[df["Predicted_Label"] != "Unknown"]

# Métricas de avaliação para sentimento
y_true_sentiment = df_filtered_sentiment["Real_Label"]
y_pred_sentiment = df_filtered_sentiment["Predicted_Label"]
sentiment_report = classification_report(y_true_sentiment, y_pred_sentiment, digits=4)
sentiment_accuracy = round(accuracy_score(y_true_sentiment, y_pred_sentiment), 4)

# Remover linhas com "Unknown" para avaliação do rating
df_filtered_rating = df[df["Predicted_Rating"] != "Unknown"]

# Métricas de avaliação para rating
y_true_rating = df_filtered_rating["Rating"].astype(float)
y_pred_rating = pd.to_numeric(df_filtered_rating["Predicted_Rating"], errors='coerce')
mae = round(mean_absolute_error(y_true_rating, y_pred_rating), 4)
mse = round(mean_squared_error(y_true_rating, y_pred_rating), 4)
r2 = round(r2_score(y_true_rating, y_pred_rating), 4)

# Imprimir as métricas
print("\n📊 Métricas de Avaliação - Sentimento:")
print(sentiment_report)
print("Accurracy (Sentimento):", sentiment_accuracy)

print("\n📊 Métricas de Avaliação - Rating:")
print(f"Erro Quadrático Absoluto: {mae}")
print(f"Erro Quadrático Médio: {mse}")
print(f"Score R²: {r2}")


# Função para upload dos ficheiros para o S3
def upload_file(local_file_path, s3_path):
    s3_client.upload_file(local_file_path, bucket_name, s3_path)
    print(f"Arquivo {local_file_path} enviado para s3://{bucket_name}/{s3_path}")


# Guardar os resultados no S3
df.to_csv("Edmunds_Car_Ratings_with_predictions_zero_shot.csv", index=False)
df.to_json("Edmunds_Car_Ratings_with_predictions_zero_shot.json", orient="records", indent=4, force_ascii=False)
upload_file('Edmunds_Car_Ratings_with_predictions_zero_shot.csv', 'output/Edmunds_Car_Ratings_with_predictions_zero_shot.csv')
upload_file('Edmunds_Car_Ratings_with_predictions_zero_shot.json', 'output/Edmunds_Car_Ratings_with_predictions_zero_shot.json')

# Guardar as métricas num ficheiro .txt
with open("metrics_report_zero_shot.txt", "w") as f:
    f.write("📊 Relatório de Classificação - Sentimento\n")
    f.write(str(sentiment_report))
    f.write(f"\nAcurácia (Sentimento): {sentiment_accuracy}\n")
    f.write("\n📊 Métricas de Regressão - Rating\n")
    f.write(f"Erro Quadrático Absoluto: {mae}\n")
    f.write(f"Erro Quadrático Médio: {mse}\n")
    f.write(f"Score R²: {r2}\n")
upload_file('metrics_report_zero_shot.txt', 'output/metrics_report_zero_shot.txt')

print("\n✅ Inferência completa. Resultados salvos em s3://{bucket_name}/output/")


📊 Métricas de Avaliação - Sentimento:
              precision    recall  f1-score   support

    Negative     0.3707    0.4149    0.3916       940
     Neutral     0.1655    0.2750    0.2066      1171
    Positive     0.9065    0.8231    0.8628      9636

    accuracy                         0.7358     11747
   macro avg     0.4809    0.5043    0.4870     11747
weighted avg     0.7898    0.7358    0.7597     11747

Accurracy (Sentimento): 0.7358

📊 Métricas de Avaliação - Rating:
Erro Quadrático Absoluto: 0.9679
Erro Quadrático Médio: 1.3378
Score R²: -0.3956
Arquivo metrics_report_zero_shot.txt enviado para s3://i32419/output/metrics_report_zero_shot.txt

✅ Inferência completa. Resultados salvos em s3://{bucket_name}/output/


In [9]:
# Ler novamente o dataset do S3
obj = s3_client.get_object(Bucket=bucket_name, Key='datasets/Edmunds_Car_Ratings_final.csv')
csv_content = obj['Body'].read().decode('utf-8')
df = pd.read_csv(StringIO(csv_content))

In [10]:
# Inferencia Few Shot

# Função para criar prompt few-shot para sentimento
def make_sentiment_prompt(df, example_indices, target_index):
    prompt = "Classify the sentiment of the following review as one of the following three words only: Positive, Neutral or Negative. Answer with exactly one of those words.\n\n"
    for idx in example_indices:
        prompt += f"Review: {df.loc[idx, 'Review']}\nSentiment: {df.loc[idx, 'Real_Label']}\n\n"
    prompt += f"Review: {df.loc[target_index, 'Review']}\nSentiment:"
    return prompt

# Função para criar prompt few-shot para rating
def make_rating_prompt(df, example_indices, target_index):
    prompt = "Predict a rating from 1.000 to 5.000, with 3 decimal places, for the following reviews. Answer with exactly four digits.\n\n"
    for idx in example_indices:
        prompt += f"Review: {df.loc[idx, 'Review']}\nRating: {df.loc[idx, 'Rating']:.3f}\n\n"
    prompt += f"Review: {df.loc[target_index, 'Review']}\nRating:"
    return prompt

# Função para criar prompt few-shot para ações
def make_action_prompt(df, example_indices, target_index):
    prompt = (
        "You are an assistant that analyzes customer reviews and determines if any action is needed.\n"
        "Respond **only** with one of the following formats, using no extra text, explanations, or repetitions. End with 'FIM'.\n\n"
        "1. If the review is entirely positive or requires no action:\n"
        "   Action: Ignorar\nFIM\n\n"
        "2. If the review contains a question, suggestion, or complaint that should be answered professionally:\n"
        "   Action: Responder\n"
        "   Response: <your short professional reply>\nFIM\n\n"
        "3. If the review indicates a serious or urgent issue:\n"
        "   Action: Escalar\n"
        "   Plan: <your short escalation plan>\nFIM\n\n"
        "4. If the review needs both an escalation and a reply (e.g., serious issue **and** the customer expects a response):\n"
        "   Action: Escalar e Responder\n"
        "   Plan: <...>\n"
        "   Response: <...>\nFIM\n\n"
        "Do not include 'Review:' or repeat the review text. Stick strictly to the format.\n"
    )

    # Exemplos few-shot (use os indices passados)
    for idx in example_indices:
        review = df.loc[idx, "Review"]

        if idx == example_indices[0]:
            response = "Action: Ignorar\nFIM"
        elif idx == example_indices[1]:
            response = (
                "Action: Responder\n"
                "Response: Thank you for sharing your experience. We're glad you found a vehicle that suits your needs!\nFIM"
            )
        elif idx == example_indices[2]:
            response = (
                "Action: Escalar e Responder\n"
                "Plan: Forward this issue to product safety and design teams for urgent review.\n"
                "Response: We appreciate you bringing this to our attention and will work to improve ventilation.\nFIM"
            )
        elif idx == example_indices[3]:
            response = (
                "Action: Escalar e Responder\n"
                "Plan: Escalate to customer service and technical support to address the issue and improve service.\n"
                "Response: We’re very sorry for your experience and appreciate your patience. Our team will be in touch to resolve this promptly.\nFIM"
            )
        elif idx == example_indices[4]:
                response = "Action: Ignorar\nFIM"

        prompt += f"Review: {review}\n{response}\n\n"

    # Prompt para review alvo
    prompt += f"Review: {df.loc[target_index, 'Review']}\nAction:"

    return prompt

In [11]:
# Invocação dos endpoints para prompts sentimento e rating
def invoke_prompt_endpoint(prompt, max_tokens=5):
    response = sagemaker_runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',
        Body=json.dumps({
            'inputs': prompt,
            'parameters': {
                'max_new_tokens': max_tokens,
                'temperature': 0.4,
            }
        })
    )
    result = response['Body'].read().decode('utf-8').strip()
    result_json = json.loads(result)
    return result_json['generated_text'].strip()

# Invocação dos endpoints para o prompt de ações (sentimento + rating)
def invoke_prompt2_endpoint(messages, max_tokens=150):
    response = sagemaker_runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',
        Body=json.dumps({
            'inputs': messages,
            'parameters': {
                'max_new_tokens': max_tokens,
                'temperature': 1,
                'top_p': 0.9,
                'stop': ["\nFIM", "\n\n", "Review:"]  # Para após FIM ou Review:
            }
        })
    )
    result = response['Body'].read().decode('utf-8').strip()
    result_json = json.loads(result)
    generated = result_json.get('generated_text', 'Unknown').strip()
    return generated

In [12]:
# Definir exemplos para prompts simples
example_indices = [0, 22, 5, 53, 49]

# Função para upload dos ficheiros para o S3
def upload_file(local_file_path, s3_path):
    s3_client.upload_file(local_file_path, bucket_name, s3_path)
    print(f"Arquivo {local_file_path} enviado para s3://{bucket_name}/{s3_path}")


# Carregar checkpoint se existir
def salvar_checkpoint(results, local_file="Edmunds_Car_Ratings_few_shots_checkpoint.json"):
    with open(local_file, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)
    upload_file(local_file, f'output/{local_file}')


local_checkpoint_file = "Edmunds_Car_Ratings_few_shots_checkpoint.json"

if os.path.exists(local_checkpoint_file):
    with open(local_checkpoint_file, "r", encoding="utf-8") as f:
        results = json.load(f)
else:
    results = []


processed_ids = {r["review_id"] for r in results}

# Iniciar colunas
df["Predicted_Label"] = None
df["Predicted_Rating"] = None
df["Action"] = None
df["Generated_Response"] = None
df["Escalation_Plan"] = None

results_batch_size = 100

# Loop de inferência (excluindo os exemplos)
for idx in df.index:
    if idx in example_indices:
        continue

    review_text = df.loc[idx, "Review"]
    review_id = str(hash(review_text))

    if review_id in processed_ids:
        continue

    # Few-shot prompt sentimento
    prompt_sentiment = make_sentiment_prompt(df, example_indices, idx)
    generated_sentiment = invoke_prompt_endpoint(prompt_sentiment)
    match = re.search(r'\b(positive|neutral|negative)\b', generated_sentiment, re.IGNORECASE)
    label = match.group(1).strip().capitalize() if match else "Unknown"

    # Few-shot prompt rating
    prompt_rating = make_rating_prompt(df, example_indices, idx)
    generated_rating = invoke_prompt_endpoint(prompt_rating)
    match_rating = re.search(r'\b(\d+\.?\d{0,3})\b', generated_rating)
    rating = float(match_rating.group(1)) if match_rating else None
    if rating is not None:
        rating = max(1.0, min(5.0, rating))
    else:
        rating = "Unknown"

    # Ação/resposta/escalar
    prompt_action = make_action_prompt(df, example_indices, idx)
    generated_action = invoke_prompt2_endpoint(prompt_action)
    
    # Ação (somente a linha da ação)
    first_line = generated_action.strip().split('\n')[0].strip().lower()
    valid_actions = ["ignorar", "responder", "escalar", "escalar e responder"]
    
    if first_line in valid_actions:
        action = first_line.title()
    else:
        action = "Unknown"
    
    # Plano (se houver)
    match_plan = re.search(r"Plan:\s*(.*?)(?:\n[A-Z][a-z]+:|\nFIM|$)", generated_action, re.IGNORECASE | re.DOTALL)
    plan = match_plan.group(1).strip() if match_plan else ""  
    
    # Resposta (se houver)
    match_response = re.search(r"Response:\s*(.*?)(?:\nFIM|$)", generated_action, re.IGNORECASE | re.DOTALL)
    response = match_response.group(1).strip() if match_response else "" 

    # Guardar
    df.at[idx, "Predicted_Label"] = label
    df.at[idx, "Predicted_Rating"] = rating
    df.at[idx, "Action"] = action
    df.at[idx, "Escalation_Plan"] = plan
    df.at[idx, "Generated_Response"] = response

    results.append({
    "review_id": review_id,
    "Predicted_Label": label,
    "Predicted_Rating": rating,
    "Action": action,
    "Escalation_Plan": plan,
    "Generated_Response": response
    })
    processed_ids.add(review_id)

    # Checkpoint a cada 100 registros
    if len(results) % results_batch_size == 0:
        print(f"Checkpoint salvo com {len(results)} registros.")
        salvar_checkpoint(results)

    print(f"[{idx+1}/{len(df)}] Review: {review_text[:50]}... -> Sentimento: {label}, Rating: {rating}, Action: {action}")
    if plan:
        print(f"  ➤ Plan: {plan}")
    if response:
        print(f"  ➤ Response: {response}")

    time.sleep(3)

# Atualizar dataframe com resultados do checkpoint
for res in results:
    matching_idxs = df[df["Review"].apply(lambda x: str(hash(x)) == res["review_id"])].index
    if not matching_idxs.empty:
        idx = matching_idxs[0]
        df.at[idx, "Predicted_Label"] = res["Predicted_Label"]
        df.at[idx, "Predicted_Rating"] = res["Predicted_Rating"]
        df.at[idx, "Action"] = res["Action"]
        df.at[idx, "Escalation_Plan"] = res["Escalation_Plan"]
        df.at[idx, "Generated_Response"] = res["Generated_Response"]

[2/11756] Review:  Rear ac blow to slow that my kid do not want to b... -> Sentimento: Negative, Rating: 2.0, Action: Escalar E Responder
  ➤ Plan: Escalate to customer service and technical support to address the issue and improve service.
  ➤ Response: We’re very sorry for your experience and appreciate your patience. Our team will be in touch to resolve this promptly.
[3/11756] Review:  This is not a small astro van type.You will need ... -> Sentimento: Positive, Rating: 4.0, Action: Ignorar


KeyboardInterrupt: 

In [13]:
# Avaliação de sentimento
df_filtered_sentiment = df[df["Predicted_Label"].notna()] #notna para tirar os valores usados nos exemplos (5 exemplos, dataset com 11 756 linhas por isso inferimos sobre 11 751)
y_true_sentiment = df_filtered_sentiment["Real_Label"]
y_pred_sentiment = df_filtered_sentiment["Predicted_Label"]
sentiment_report = classification_report(y_true_sentiment, y_pred_sentiment, digits=4)
sentiment_accuracy = round(accuracy_score(y_true_sentiment, y_pred_sentiment), 4)

# Avalicação de rating
df_filtered_rating = df[df["Predicted_Label"].notna()] #notna para tirar os valores usados nos exemplos (5 exemplos, dataset com 11 756 linhas por isso inferimos sobre 11 751)
y_true_rating = df_filtered_rating["Rating"].astype(float)
y_pred_rating = pd.to_numeric(df_filtered_rating["Predicted_Rating"], errors='coerce')
mae = round(mean_absolute_error(y_true_rating, y_pred_rating), 4)
mse = round(mean_squared_error(y_true_rating, y_pred_rating), 4)
r2 = round(r2_score(y_true_rating, y_pred_rating), 4)

# Print resultados
print("\n📊 Métricas de Avaliação Few Shots - Sentimento:")
print(sentiment_report)
print("Accuracy (Sentimento):", sentiment_accuracy)

print("\n📊 Métricas de Avaliação Few Shots - Rating:")
print(f"Erro Quadrático Absoluto: {mae}")
print(f"Erro Quadrático Médio: {mse}")
print(f"Score R²: {r2}")

# Exportar resultados para csv e S3
df.to_csv("Edmunds_Car_Ratings_with_predictions_few_shots.csv", index=False)
df.to_json("Edmunds_Car_Ratings_with_predictions_few_shots.json", orient="records", indent=4, force_ascii=False)
upload_file('Edmunds_Car_Ratings_with_predictions_few_shots.csv', 'output/Edmunds_Car_Ratings_with_predictions_few_shots.csv')
upload_file('Edmunds_Car_Ratings_with_predictions_few_shots.json', 'output/Edmunds_Car_Ratings_with_predictions_few_shots.json')

with open("metrics_report_few_shots.txt", "w", encoding="utf-8") as f:
    f.write("📊 Relatório de Classificação - Sentimento\n")
    f.write(str(sentiment_report))
    f.write(f"Accuracy (Sentimento): {sentiment_accuracy}\n")
    f.write("\n📊 Métricas de Regressão - Rating\n")
    f.write(f"Erro Quadrático Absoluto: {mae}")
    f.write(f"Erro Quadrático Médio: {mse}\n")
    f.write(f"Score R²: {r2}\n")
upload_file('metrics_report_few_shots.txt', 'output/metrics_report_few_shots.txt')

print(f"\n✅ Inferência completa. Resultados salvos em s3://{bucket_name}/output/")

/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_


📊 Métricas de Avaliação Few Shots - Sentimento:
              precision    recall  f1-score   support

    Negative     0.0000    0.0000    0.0000         0
     Neutral     0.0000    0.0000    0.0000         1
    Positive     1.0000    1.0000    1.0000         1

    accuracy                         0.5000         2
   macro avg     0.3333    0.3333    0.3333         2
weighted avg     0.5000    0.5000    0.5000         2

Accuracy (Sentimento): 0.5

📊 Métricas de Avaliação Few Shots - Rating:
Erro Quadrático Médio: 1.0
Erro Quadrático Médio: 1.0
Score R²: 0.0
Arquivo Edmunds_Car_Ratings_with_predictions_few_shots.csv enviado para s3://i32419/output/Edmunds_Car_Ratings_with_predictions_few_shots.csv
Arquivo Edmunds_Car_Ratings_with_predictions_few_shots.json enviado para s3://i32419/output/Edmunds_Car_Ratings_with_predictions_few_shots.json
Arquivo metrics_report_few_shots.txt enviado para s3://i32419/output/metrics_report_few_shots.txt

✅ Inferência completa. Resultados salvos em s